# 28 · Calibration and selective prediction on base-rate initialised heads

Design log
- Notebook 27 heads (ResNet-50 frozen, base-rate initialisation, seed 0) have lower log-loss and higher top-1 endorsement than the frequency baseline. Notebook 21 calibration and selection figures were measured on default-initialised heads and do not describe these heads.
- The selected run per head is the one with the lowest validation loss in notebook 27, the same rule notebook 27 used.
- Calibrators are fitted on the calibration partition and compared on validation, as in notebook 21. Temperature scaling is one global value, so it cannot change the chosen label or the confidence ordering. Sigmoid scaling is fitted per label and can change both.
- Changes in log-loss and ECE against the uncalibrated head carry participant-group bootstrap intervals, 1000 replicates, with every calibrator scored on the same draw.
- Selective prediction uses the highest calibrated probability as confidence. Thresholds for each target coverage are set on validation, and panel disagreement among assessable answers is reported with participant-group intervals, as in notebook 21.
- Validation was used for early stopping and learning-rate selection, so absolute validation figures are slightly optimistic. Comparisons between calibrators on the same head are not affected. Test records are not read.

In [ ]:
from pathlib import Path
import os, sys, json
ON_COLAB = "google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG"))
if ON_COLAB and not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")
ROOT = Path(os.environ.get("ONCOPLATE_DRIVE_ROOT", "/content/drive/MyDrive/OncoPlate_Research"))
pointer = ROOT / ".oncoplate_install.json"
preferred = json.loads(pointer.read_text())["repository_path"] if pointer.exists() else str(ROOT / "oncoplate-research")
REPO = Path(os.environ.get("ONCOPLATE_REPO", preferred))
if not (REPO / "src/oncoplate").exists():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "src/oncoplate").exists(): REPO = candidate; break
assert (REPO / "src/oncoplate").exists(), "Run the supplied installer notebook or set ONCOPLATE_REPO to the extracted repository."
sys.path.insert(0, str(REPO / "src"))
from oncoplate.config import load_config, paths, initialize
from oncoplate.io import read_json, write_json, read_table, write_table, read_jsonl, write_jsonl, utcnow
cfg = load_config(REPO, root=ROOT, mode=os.environ.get("ONCOPLATE_MODE", "research"))
p = paths(cfg)
print("Dataset:", cfg["study"]["dataset"], "| Mode:", cfg["mode"], "| Persistent root:", cfg["root"])


## 1. Selected base-rate heads and their predictions

In [ ]:
import numpy as np, pandas as pd
from oncoplate.pipeline import frequency_baseline, load_study, prediction_arrays
from oncoplate.training import predict_run
from oncoplate.calibration import calibration_metrics, probabilities, fit_temperature, fit_sigmoid
from oncoplate.statistics import analysis_weights, foundation_group_intervals
from oncoplate.selection import choose_threshold

HEADS = ("independent", "joint")
STUDY = {h: load_study(cfg, h, stage_images=True) for h in HEADS}
PREVALENCE = {h: frequency_baseline(cfg, h, "validation")["probabilities"][0] for h in HEADS}

def evaluate(prob, y, mask, ids, head):
    m = calibration_metrics(prob, y, mask)
    sub = STUDY[head][0].set_index("record_id").loc[ids].reset_index()
    w = analysis_weights(sub, {"meal": 1.0})
    rows = np.arange(len(prob)); j = prob.argmax(1)
    observed = mask[rows, j] > 0; den = w[observed].sum()
    top1 = float(np.sum(w[observed] * y[rows, j][observed]) / den) if den else float("nan")
    return {"log_loss": m["log_loss"], "ece": m["ece"], "top1_endorsement": top1}

def frequency_on(ids, y, mask, head):
    # Scored on exactly the records, targets and masks the image model is scored on.
    return evaluate(np.broadcast_to(PREVALENCE[head], y.shape).copy(), y, mask, ids, head)

# read_table loads every column as text; numeric types are needed to sort by loss.
grid = pd.read_csv(p["reports"] / "foundation_prior_init_grid.csv")
chosen = grid.sort_values(["head", "best_validation_bce", "lr"]).groupby("head").head(1)
PRED = {}
for r in chosen.itertuples():
    records, targets = STUDY[r.head]
    run = Path(r.run_dir)
    PRED[r.head] = {"run": run, "lr": float(r.lr),
                    "validation": prediction_arrays(run / "validation_predictions.npz"),
                    "calibration": predict_run(run, records[records.split.eq("calibration")], targets, p["features"])}
pd.DataFrame([{"head": k, "run": v["run"].name, "validation_records": len(v["validation"]["ids"]),
               "calibration_records": len(v["calibration"]["ids"])} for k, v in PRED.items()])

## 2. Calibrators fitted on the calibration partition, compared on validation

In [ ]:
CAL, rows = {}, []
for head, d in PRED.items():
    c, v = d["calibration"], d["validation"]
    CAL[head] = {"identity": {"kind": "identity"},
                 "temperature": fit_temperature(c["logits"], c["y"], c["mask"]),
                 "sigmoid": fit_sigmoid(c["logits"], c["y"], c["mask"])}
    write_json(d["run"] / "calibrators.json", CAL[head])
    rows.append({"head": head, "calibrator": "frequency baseline (no image)", **frequency_on(v["ids"], v["y"], v["mask"], head)})
    for name, calib in CAL[head].items():
        rows.append({"head": head, "calibrator": name,
                     **evaluate(probabilities(v["logits"], calib), v["y"], v["mask"], v["ids"], head)})
calibration_table = pd.DataFrame(rows)
write_table(p["reports"] / "foundation_prior_init_calibration_validation.csv", calibration_table)
print("fitted temperature:", {h: round(CAL[h]["temperature"]["temperature"], 4) for h in HEADS})
calibration_table

## 3. Participant-group intervals for the calibration changes

In [ ]:
def paired_group_bootstrap(head, probs, y, mask, ids, B=1000, seed=0):
    # Resamples participant groups; every calibrator is scored on the same draw.
    groups = STUDY[head][0].set_index("record_id").loc[ids, "group_id"].to_numpy()
    members = [np.flatnonzero(groups == g) for g in np.unique(groups)]
    rng = np.random.default_rng(seed); draws = []
    for _ in range(B):
        pick = np.concatenate([members[i] for i in rng.integers(0, len(members), len(members))])
        m = {k: calibration_metrics(v[pick], y[pick], mask[pick]) for k, v in probs.items()}
        draws.append({f"{k}_{s}": m[k][s] - m["identity"][s] for k in probs if k != "identity" for s in ("log_loss", "ece")})
    return pd.DataFrame(draws)

rows = []
for head, d in PRED.items():
    v = d["validation"]
    probs = {k: probabilities(v["logits"], c) for k, c in CAL[head].items()}
    point = {k: calibration_metrics(pr, v["y"], v["mask"]) for k, pr in probs.items()}
    boot = paired_group_bootstrap(head, probs, v["y"], v["mask"], v["ids"])
    for k in ("temperature", "sigmoid"):
        for s in ("log_loss", "ece"):
            lo, hi = np.quantile(boot[f"{k}_{s}"], [0.025, 0.975])
            rows.append({"head": head, "calibrator": k, "metric": s,
                         "change_vs_uncalibrated": point[k][s] - point["identity"][s],
                         "ci95_low": float(lo), "ci95_high": float(hi), "ci_excludes_zero": bool(lo > 0 or hi < 0)})
gains = pd.DataFrame(rows)
write_table(p["reports"] / "foundation_prior_init_calibration_gains.csv", gains)
gains

## 4. Selective prediction on validation

In [ ]:
TARGETS = (1.0, 0.9, 0.8, 0.7, 0.6, 0.5)
rows = []
for head, d in PRED.items():
    v = d["validation"]; n = len(v["ids"]); idx = np.arange(n)
    sub = STUDY[head][0].set_index("record_id").loc[v["ids"]].reset_index()
    w = analysis_weights(sub, {"meal": 1.0})
    for name in ("identity", "temperature", "sigmoid"):
        prob = probabilities(v["logits"], CAL[head][name]); j = prob.argmax(1); conf = prob.max(1)
        observed = v["mask"][idx, j] > 0; agreement = v["y"][idx, j]
        for target in TARGETS:
            thr = choose_threshold(conf, np.ones(n, bool), target, w)
            frame = pd.DataFrame({"record_id": v["ids"], "group_id": sub["group_id"].to_numpy(),
                                  "accepted": conf >= thr["threshold"], "observed": observed, "agreement": agreement})
            est, _ = foundation_group_intervals(frame, B=1000)
            dis = est["metrics"]["panel_disagreement_among_assessable_answers"]
            rows.append({"head": head, "calibrator": name, "target_coverage": target,
                         "coverage": est["metrics"]["answer_coverage_all_records"]["estimate"],
                         "panel_disagreement": dis["estimate"], "ci95_low": dis["ci95"][0], "ci95_high": dis["ci95"][1]})
selective = pd.DataFrame(rows)
write_table(p["reports"] / "foundation_prior_init_selective_validation.csv", selective)
selective

## 5. Summary

In [ ]:
def point_ci(frame, **keys):
    row = frame.loc[np.logical_and.reduce([np.isclose(frame[k], v) if isinstance(v, float) else frame[k].eq(v)
                                           for k, v in keys.items()])].iloc[0]
    value = row["change_vs_uncalibrated"] if "change_vs_uncalibrated" in row else row["panel_disagreement"]
    return [round(float(value), 4), round(float(row["ci95_low"]), 4), round(float(row["ci95_high"]), 4)]

verdict = {}
for head in HEADS:
    verdict[head] = {
        "run": PRED[head]["run"].name, "temperature": round(float(CAL[head]["temperature"]["temperature"]), 4),
        "log_loss_change_temperature": point_ci(gains, head=head, calibrator="temperature", metric="log_loss"),
        "log_loss_change_sigmoid": point_ci(gains, head=head, calibrator="sigmoid", metric="log_loss"),
        "ece_change_sigmoid": point_ci(gains, head=head, calibrator="sigmoid", metric="ece"),
        "disagreement_full_coverage_identity": point_ci(selective, head=head, calibrator="identity", target_coverage=1.0),
        "disagreement_80pct_coverage_identity": point_ci(selective, head=head, calibrator="identity", target_coverage=0.8),
        "disagreement_80pct_coverage_sigmoid": point_ci(selective, head=head, calibrator="sigmoid", target_coverage=0.8)}
write_json(p["reports"] / "foundation_prior_init_calibration_summary.json", verdict)
print(json.dumps(verdict, indent=2))